In [4]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator
from sklearn.base import ClassifierMixin

from sklearn.model_selection import BaseCrossValidator

from sklearn.utils.validation import check_X_y, check_is_fitted
from sklearn.utils.validation import check_array
from sklearn.utils.multiclass import check_classification_targets
from sklearn.metrics.pairwise import pairwise_distances

import pytest
import numpy as np
import pandas as pd
from numpy.testing import assert_array_equal

from sklearn.utils.estimator_checks import check_estimator
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.datasets import make_classification
from sklearn.neighbors import KNeighborsClassifier



In [13]:
class KNearestNeighbors(BaseEstimator, ClassifierMixin):
    """KNearestNeighbors classifier."""

    def __init__(self, n_neighbors=1):
        self.n_neighbors = n_neighbors

    def fit(self, X, y):
        """Fit the model using X as training data and y as target values."""
        X, y = check_X_y(X, y, ensure_2d=True, dtype="numeric")
        self.X_ = X
        self.y_ = y.astype(int)  # Ensure labels are integers
        return self

    def predict(self, X):
        """Predict the class labels for the provided data."""
        check_is_fitted(self, ['X_', 'y_'])
        X = check_array(X, ensure_2d=True, dtype="numeric")

        # Compute distances
        distances = pairwise_distances(X, self.X_)

        # Find the nearest neighbor indices
        if self.n_neighbors == 1:
            nearest_indices = np.argmin(distances, axis=1)
        else:
            nearest_indices = np.argsort(distances, axis=1)[:, :self.n_neighbors]

        # Predict using the nearest neighbor labels
        if self.n_neighbors == 1:
            return self.y_[nearest_indices]
        else:
            # Aggregate labels and return the most common one
            return np.apply_along_axis(lambda idxs: np.bincount(self.y_[idxs]).argmax(), axis=1, arr=nearest_indices)

    def score(self, X, y):
        """Return the mean accuracy on the given test data and labels."""
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

In [17]:
@pytest.mark.parametrize("k", [1, 3, 5, 7])
def test_one_nearest_neighbor_match_sklearn(k):
    X, y = make_classification(n_samples=200, n_features=20,
                               random_state=42)
    X_train, X_test, y_train, y_test = \
        train_test_split(X, y, random_state=42)
    knn = KNeighborsClassifier(n_neighbors=k)
    y_pred_sk = knn.fit(X_train, y_train).predict(X_test)

    onn = KNearestNeighbors(k)
    y_pred_me = onn.fit(X_train, y_train).predict(X_test)
    assert_array_equal(y_pred_me, y_pred_sk)

    assert onn.score(X_test, y_test) == knn.score(X_test, y_test)


@pytest.mark.parametrize("k", [1, 3, 5, 7])
def test_one_nearest_neighbor_check_estimator(k):
    check_estimator(KNearestNeighbors(n_neighbors=k))
